<a href="https://colab.research.google.com/github/saltsoup/Mavic-Reccomendations/blob/main/Siqian_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BT4222 Group 1 — JDSearch Personalised Re-ranking
**Data paths (Google Drive, unchanged from our original setup):**
- `user_df`  ← `/content/drive/MyDrive/BT4222 Final/JDSearch/Raw Data/user_behavior_data.txt`
- `product_df` (raw) ← `/content/drive/MyDrive/BT4222 Final/JDSearch/Raw Data/product_meta_data.txt`
- product chunk cache ← `/content/drive/MyDrive/BT4222 Final/JDSearch/Raw Data/product_chunks/`
- saved model artifacts ← `/content/drive/MyDrive/BT4222 Final/JDSearch/model_artifacts/` (new — saved to Drive this time so a Colab crash/disconnect does not wipe trained models)


## Setup

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import gc
import glob
import pickle
from pathlib import Path
from collections import Counter, defaultdict
import matplotlib.pyplot as plt

BASE = Path("/content/drive/MyDrive/BT4222 Final/JDSearch/Raw Data")
USER_TXT = BASE / "user_behavior_data.txt"
PRODUCT_TXT = BASE / "product_meta_data.txt"
CHUNKS_DIR = BASE / "product_chunks"
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

# New: dedicated folder for saved model artifacts (persists on Drive across Colab sessions/crashes)
ARTIFACTS_DIR = BASE.parent / "model_artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


Mounted at /content/drive


##Load `user_df`

In [3]:
user_df = pd.read_csv(USER_TXT, sep="\t", dtype=str, low_memory=False)
print(user_df.shape)


(173831, 7)


In [4]:
user_df.columns.tolist()


['query',
 'candidate_wid_list',
 'candidate_label_list',
 'history_qry_list',
 'history_wid_list',
 'history_type_list',
 'history_time_list']

In [10]:
# ============================================================
# Parse candidate IDs and labels
# Must run before any code using _parsed columns
# ============================================================

import pandas as pd

def parse_candidate_ids(x):
    if pd.isna(x) or x == '':
        return []
    return str(x).split('_')

def parse_candidate_labels(x):
    if pd.isna(x) or x == '':
        return []
    return [int(float(v)) for v in str(x).split('_')]

user_df['candidate_wid_list_parsed'] = (
    user_df['candidate_wid_list'].apply(parse_candidate_ids)
)

user_df['candidate_label_list_parsed'] = (
    user_df['candidate_label_list'].apply(parse_candidate_labels)
)

# Calculate slate length
user_df['slate_len'] = (
    user_df['candidate_wid_list_parsed'].map(len)
)

# Verify everything is aligned
assert (
    user_df['candidate_wid_list_parsed'].map(len)
    == user_df['candidate_label_list_parsed'].map(len)
).all()

print("Candidate parsing successful!")
print("Slate lengths calculated!")
print("Candidate-label alignment verified!")

Candidate parsing successful!
Slate lengths calculated!
Candidate-label alignment verified!


### Inspecting user_df

In [5]:
# STEP 1: Inspect user_df
print("Shape:", user_df.shape)

print("\nColumns:")
print(user_df.columns.tolist())

print("\nData types:")
print(user_df.dtypes)

print("\nFirst 3 rows:")
display(user_df.head(3))


Shape: (173831, 7)

Columns:
['query', 'candidate_wid_list', 'candidate_label_list', 'history_qry_list', 'history_wid_list', 'history_type_list', 'history_time_list']

Data types:
query                   object
candidate_wid_list      object
candidate_label_list    object
history_qry_list        object
history_wid_list        object
history_type_list       object
history_time_list       object
dtype: object

First 3 rows:


,query,candidate_wid_list,candidate_label_list,history_qry_list,history_wid_list,history_type_list,history_time_list
0,63995226602574196100244625575117,36098226_17729172_34921664_68317377_693764_360...,1.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0....,-1_-1_-1_68784030_36547097602574193663289_95...,9227588_52788392_4585436_87422901_57364733_210...,ORD_ORD_CART_CART_CART_CART_CART_CART_CART_CAR...,0_0_22967659_82448_432_10315_51_188_51_254173_...
1,1254963427438884,30857980_44577431_1485384_63372823_19754701_29...,1.0_1.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0....,9755613572694219_46916741_-1_-1_-1_-1_-1_4927...,18256630_46206270_65230659_86878921_32743602_5...,ORD_ORD_ORD_CART_CART_CART_CART_ORD_CART_CART_...,0_4813478_1115463_14061983_920_467229_304_1197...
2,24374376067768939206305,42155506_90092899_4336287_74203839_39478459_37...,1.0_1.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0.0_0....,-1_-1,8133925_59478082,CLICK_CLICK,0_27_49660


In [6]:
user_df['slate_len'] = user_df['candidate_wid_list'].str.count('_') + 1
print(user_df['slate_len'].describe())

print("Sessions with slate length = 1:", (user_df['slate_len'] == 1).sum())
user_df[user_df['slate_len'] == 1][['query', 'candidate_wid_list', 'candidate_label_list']].head()


count    173831.000000
mean         89.224661
std          38.857357
min           1.000000
25%          76.000000
50%          90.000000
75%          95.000000
max         200.000000
Name: slate_len, dtype: float64
Sessions with slate length = 1: 487


,query,candidate_wid_list,candidate_label_list
177,71300425679728828855555792469075742895467...,78361212,1.0
200,52027478278253308005728059056538602574196...,3967722,2.0
422,69207248707673507936922256796994995516589...,27724985,2.0
1147,96436578367139964145559778453805845804104...,91801771,0.0
1685,98243753373984259012786264319596436578366...,45425822,2.0


##Parsing the candidate lists


In [7]:
# Candidate ID parsing

def parse_candidate_ids(x):
    if pd.isna(x) or x == '':
        return []

    return str(x).split('_')

user_df['candidate_wid_list_parsed'] = (
    user_df['candidate_wid_list'].apply(parse_candidate_ids)
)

In [8]:
# STEP 2A: Check parsed columns

required_cols = [
    'candidate_wid_list_parsed',
    'candidate_label_list_parsed'
]

missing_cols = [
    c for c in required_cols
    if c not in user_df.columns
]

print("Missing parsed columns:", missing_cols)

Missing parsed columns: ['candidate_label_list_parsed']


##Check candidate-label alignment
Every candidate should have exactly one label.

In [11]:
# Check that every candidate has a corresponding label

candidate_lengths = user_df[
    'candidate_wid_list_parsed'
].apply(len)

label_lengths = user_df[
    'candidate_label_list_parsed'
].apply(len)

lengths_match = candidate_lengths.eq(label_lengths)

print("Total sessions:", len(user_df))

print("Sessions with matching lengths:",
      lengths_match.sum())

print("Sessions with mismatched lengths:",
      (~lengths_match).sum())

print("Minimum candidates:", candidate_lengths.min())
print("Maximum candidates:", candidate_lengths.max())

Total sessions: 173831
Sessions with matching lengths: 173831
Sessions with mismatched lengths: 0
Minimum candidates: 1
Maximum candidates: 200


## Identify eligible sessions
For filtering out All-negative and All-positive sessions

In [12]:
# Identify sessions with both positive and negative candidates

def check_session_eligibility(labels):
    labels = [int(x) for x in labels]

    has_positive = any(x > 0 for x in labels)
    has_negative = any(x == 0 for x in labels)

    return has_positive, has_negative


eligibility = user_df[
    'candidate_label_list_parsed'
].apply(check_session_eligibility)

has_positive = eligibility.apply(lambda x: x[0])
has_negative = eligibility.apply(lambda x: x[1])

eligible_mask = has_positive & has_negative

print("Eligible sessions:", eligible_mask.sum())
print("All-negative sessions:", (~has_positive).sum())
print("No-negative sessions:", (~has_negative).sum())

Eligible sessions: 171175
All-negative sessions: 2103
No-negative sessions: 553


In [13]:
# Keep only sessions with both positive and negative candidates

user_clean = user_df.loc[eligible_mask].copy()

print("Original sessions:", len(user_df))
print("Cleaned sessions:", len(user_clean))
print("Excluded sessions:", len(user_df) - len(user_clean))

Original sessions: 173831
Cleaned sessions: 171175
Excluded sessions: 2656


##Create 80/10/10 split
80/10/10 split for training, validation and testing

In [14]:
# Preserve original row numbers as session identifiers

user_clean['session_id'] = user_clean.index

print("Unique session IDs:",
      user_clean['session_id'].nunique())

print("Total sessions:", len(user_clean))

assert user_clean['session_id'].is_unique

Unique session IDs: 171175
Total sessions: 171175


In [19]:
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42

# First split: 80% training, 20% temporary
train_ids, temp_ids = train_test_split(
    user_clean['session_id'],
    test_size=0.20,
    random_state=RANDOM_SEED,
    shuffle=True
)

# Second split: 10% validation, 10% testing
val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=RANDOM_SEED,
    shuffle=True
)

# Assign each session to its split
user_clean['split'] = 'unassigned'

user_clean.loc[
    user_clean['session_id'].isin(train_ids), 'split'
] = 'train'

user_clean.loc[
    user_clean['session_id'].isin(val_ids), 'split'
] = 'val'

user_clean.loc[
    user_clean['session_id'].isin(test_ids), 'split'
] = 'test'

print(user_clean['split'].value_counts())

split
train    136940
test      17118
val       17117
Name: count, dtype: int64


In [24]:
# Verify the split
# Check that the three groups do not overlap

assert set(train_ids).isdisjoint(set(val_ids))
assert set(train_ids).isdisjoint(set(test_ids))
assert set(val_ids).isdisjoint(set(test_ids))

# Check every session was assigned
assert (user_clean['split'] != 'unassigned').all()

print("Split verification passed!")

print("\nSplit proportions:")
print(user_clean['split'].value_counts(normalize=True))

Split verification passed!

Split proportions:
split
train    0.800000
test     0.100003
val      0.099997
Name: proportion, dtype: float64


### Save split data into Google Drive

In [25]:

# Main project directory
PROJECT_DIR = Path(
    "/content/drive/MyDrive/BT4222 Final/JDSearch"
)

# Folder for cleaned and processed datasets
OUTPUT_DIR = PROJECT_DIR / "processed"

# Create folder if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save user split
user_split = user_clean[
    ['session_id', 'split']
].copy()

user_split.to_csv(
    OUTPUT_DIR / "user_split.csv",
    index=False
)

print("Saved successfully!")
print("File location:", OUTPUT_DIR / "user_split.csv")

Saved successfully!
File location: /content/drive/MyDrive/BT4222 Final/JDSearch/processed/user_split.csv


## Build candidate-level dataset
Currently, user_clean contains one row per search session, with multiple candidate products stored in lists.

For LightGBM training, we need one row per candidate product.

Note for team: The candidate-level dataset can be used by person 2, 3 and 4 to add features and train the model.

In [26]:
# STEP 4A: Validate candidate lists
required_cols = [
    'session_id',
    'split',
    'candidate_wid_list_parsed',
    'candidate_label_list_parsed'
]

assert all(c in user_clean.columns for c in required_cols)

id_lengths = user_clean['candidate_wid_list_parsed'].map(len)
label_lengths = user_clean['candidate_label_list_parsed'].map(len)

assert id_lengths.eq(label_lengths).all(), \
    "Candidate IDs and labels are misaligned!"

print("Sessions:", len(user_clean))
print("Total candidate rows expected:", id_lengths.sum())
print("Average candidates per session:", id_lengths.mean())
print("All candidate lists aligned!")

Sessions: 171175
Total candidate rows expected: 15348049
Average candidates per session: 89.66291222433182
All candidate lists aligned!


### Expand candidate lists in memory-efficient batches
- Takes 2,000 sessions at a time
- Expands each session's candidate IDs and labels together.
- Converts them into individual rows.
- Saves each batch as Parquet.
- Releases memory before processing the next batch.

In [27]:
# STEP 4B: Expand candidate lists in memory-efficient batches

CANDIDATE_DIR = OUTPUT_DIR / "candidate_chunks"
CANDIDATE_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 2000

# Keep only the columns required for the base table
cols = [
    'session_id',
    'split',
    'candidate_wid_list_parsed',
    'candidate_label_list_parsed'
]

total_rows = 0

for batch_num, start in enumerate(
    range(0, len(user_clean), BATCH_SIZE)
):

    batch = user_clean.iloc[
        start:start + BATCH_SIZE
    ][cols]

    # Expand candidate IDs and labels together
    expanded = batch.explode(
        [
            'candidate_wid_list_parsed',
            'candidate_label_list_parsed'
        ],
        ignore_index=True
    )

    # Rename columns
    expanded = expanded.rename(columns={
        'candidate_wid_list_parsed': 'wid',
        'candidate_label_list_parsed': 'label'
    })

    # Standardize data types
    expanded['wid'] = expanded['wid'].astype(str)
    expanded['label'] = expanded['label'].astype('int8')

    # Save batch
    output_file = (
        CANDIDATE_DIR /
        f"candidate_batch_{batch_num:03d}.parquet"
    )

    expanded.to_parquet(
        output_file,
        index=False
    )

    total_rows += len(expanded)

    if batch_num % 10 == 0:
        print(
            f"Batch {batch_num}: "
            f"{total_rows:,} rows saved"
        )

    del batch, expanded
    gc.collect()

print("\nCompleted!")
print("Total candidate rows saved:", total_rows)
print("Files saved to:", CANDIDATE_DIR)

Batch 0: 179,397 rows saved
Batch 10: 1,965,899 rows saved
Batch 20: 3,760,666 rows saved
Batch 30: 5,552,022 rows saved
Batch 40: 7,340,957 rows saved
Batch 50: 9,141,888 rows saved
Batch 60: 10,928,635 rows saved
Batch 70: 12,719,371 rows saved
Batch 80: 14,520,907 rows saved

Completed!
Total candidate rows saved: 15348049
Files saved to: /content/drive/MyDrive/BT4222 Final/JDSearch/processed/candidate_chunks


In [ ]:
# STEP 4C: Verify saved candidate chunks

import glob

chunk_files = sorted(
    glob.glob(str(CANDIDATE_DIR / "*.parquet"))
)

print("Number of chunks:", len(chunk_files))

saved_rows = 0

for file in chunk_files:
    df = pd.read_parquet(
        file,
        columns=['session_id', 'wid', 'label', 'split']
    )

    saved_rows += len(df)

    del df

print("Total saved rows:", saved_rows)
print("Expected rows:", id_lengths.sum())

assert saved_rows == id_lengths.sum()

print("Candidate expansion verified!")

In [ ]:
# Shuffle candidate chunks with updating progress

from IPython.display import clear_output

SHUFFLED_DIR = OUTPUT_DIR / "candidate_chunks_shuffled"
SHUFFLED_DIR.mkdir(parents=True, exist_ok=True)

chunk_files = sorted(CANDIDATE_DIR.glob("*.parquet"))

for chunk_num, file in enumerate(chunk_files):

    df = pd.read_parquet(file)

    # Shuffle candidates within each session
    df = (
        df.groupby('session_id', group_keys=False, sort=False)
          .sample(frac=1, random_state=42)
          .reset_index(drop=True)
    )

    # Save shuffled chunk
    output_file = SHUFFLED_DIR / file.name
    df.to_parquet(output_file, index=False)

    # Replace previous progress message
    clear_output(wait=True)
    print(f"Shuffling candidate chunks: {chunk_num + 1}/{len(chunk_files)}")

print("\nAll candidate chunks shuffled successfully!")

##Parse historical interaction data


In [ ]:
# Parse historical interaction data

def parse_history(x):
    if pd.isna(x) or x == '':
        return []
    return str(x).split('_')

# Parse historical queries
user_clean['history_qry_list_parsed'] = (
    user_clean['history_qry_list'].apply(parse_history)
)

# Parse historical product IDs
user_clean['history_wid_list_parsed'] = (
    user_clean['history_wid_list'].apply(parse_history)
)

# Parse interaction types
user_clean['history_type_list_parsed'] = (
    user_clean['history_type_list'].apply(parse_history)
)

# Parse time intervals into integers (seconds)
user_clean['history_time_list_parsed'] = (
    user_clean['history_time_list'].apply(
        lambda x: [int(v) for v in parse_history(x)]
    )
)

print("Historical columns parsed successfully!")

In [ ]:
# Check parsed history columns

parsed_cols = [
    'history_qry_list_parsed',
    'history_wid_list_parsed',
    'history_type_list_parsed',
    'history_time_list_parsed'
]

for col in parsed_cols:
    print(
        col,
        "EXISTS" if col in user_clean.columns else "MISSING"
    )

## Convert time intervals into elapsed days
Investigate the 1,200 cart actions beyond 90 days

In [ ]:
# Reconstruct elapsed days before current query
def reconstruct_elapsed_days(time_list):
    """
    Convert consecutive time intervals into elapsed days
    before the current test query.

    Assumes:
    - History actions are ordered oldest to newest.
    - len(time_list) = number of actions + 1.
    - time_list[0] is not an action-to-query interval.
    """
    gaps = np.asarray(time_list[1:], dtype=np.float64)

    elapsed_seconds = np.cumsum(gaps[::-1])[::-1]

    return elapsed_seconds / 86400


# Test on a small example
example = [0, 100, 200, 300]

print(reconstruct_elapsed_days(example) * 86400)

In [ ]:
user_clean['history_elapsed_days'] = (
    user_clean['history_time_list_parsed']
    .apply(reconstruct_elapsed_days)
)

print("Elapsed days reconstructed!")

print("\nFirst session:")
print(
    user_clean['history_elapsed_days'].iloc[0][:10]
)

In [ ]:
# Investigate cart timing outliers

cart_over_90 = 0
cart_total = 0
affected_sessions = 0

for types, days in zip(
    user_clean['history_type_list_parsed'],
    user_clean['history_elapsed_days']
):
    types_arr = np.asarray(types)
    days_arr = np.asarray(days)

    cart_mask = types_arr == 'CART'

    cart_total += cart_mask.sum()

    n_outliers = np.sum(
        cart_mask & (days_arr > 90)
    )

    cart_over_90 += n_outliers

    if n_outliers > 0:
        affected_sessions += 1

print("Total cart actions:", cart_total)
print("Cart actions older than 90 days:", cart_over_90)
print("Affected sessions:", affected_sessions)

if cart_total > 0:
    print(
        "Percentage of carts older than 90 days:",
        f"{cart_over_90 / cart_total:.4%}"
    )

In [ ]:
# Count interaction types by time window
windows = {
    '0-7d': Counter(),
    '7-90d': Counter(),
    '90-365d': Counter(),
    '365+d': Counter()
}

for types, days in zip(
    user_clean['history_type_list_parsed'],
    user_clean['history_elapsed_days']
):
    for action, day in zip(types, days):

        if day < 7:
            window = '0-7d'
        elif day < 90:
            window = '7-90d'
        elif day < 365:
            window = '90-365d'
        else:
            window = '365+d'

        windows[window][action] += 1

window_counts = pd.DataFrame(windows).T.fillna(0).astype(int)

print("Historical interaction counts:")
display(window_counts)

print("\nPercentage within each window:")
display(
    window_counts.div(
        window_counts.sum(axis=1),
        axis=0
    ).mul(100).round(2)
)

## Exclude cart actions older than 90 days

In [ ]:
# Exclude cart actions older than 90 days
# Preserve original history columns

def clean_history(row):
    wids = row['history_wid_list_parsed']
    types = row['history_type_list_parsed']
    days = row['history_elapsed_days']

    keep = [
        not (action == 'CART' and day > 90)
        for action, day in zip(types, days)
    ]

    return (
        [wid for wid, k in zip(wids, keep) if k],
        [action for action, k in zip(types, keep) if k],
        [day for day, k in zip(days, keep) if k]
    )

cleaned = user_clean.apply(clean_history, axis=1)

user_clean['history_wid_clean'] = cleaned.apply(lambda x: x[0])
user_clean['history_type_clean'] = cleaned.apply(lambda x: x[1])
user_clean['history_days_clean'] = cleaned.apply(lambda x: x[2])

print("Cleaned history columns created!")

## Create session-level recency features
Create 3 recency features based on time window
- 0-70 days -> Recent activity
- 7-90 days -> Medium-term activity
- 90-365 days -> Long-term purchasing history



In [ ]:
# Engineer session-level recency features

from collections import Counter
from tqdm.auto import tqdm

def create_recency_features(types, days):

    features = {}

    windows = {
        '0_7d': (0, 7),
        '7_90d': (7, 90),
        '90_365d': (90, 365)
    }

    for window_name, (start, end) in windows.items():

        actions_in_window = [
            action
            for action, day in zip(types, days)
            if start <= day < end
        ]

        counts = Counter(actions_in_window)

        for action in ['CLICK', 'CART', 'ORD', 'FLW']:
            features[
                f'{action.lower()}_count_{window_name}'
            ] = counts.get(action, 0)

    # Overall history information
    features['history_len_clean'] = len(types)

    features['history_span_days'] = (
        max(days) if len(days) > 0 else np.nan
    )

    features['days_since_last_interaction'] = (
        min(days) if len(days) > 0 else np.nan
    )

    return features


# Generate features
recency_records = []

for session_id, types, days in tqdm(
    zip(
        user_clean['session_id'],
        user_clean['history_type_clean'],
        user_clean['history_days_clean']
    ),
    total=len(user_clean),
    desc="Creating recency features"
):

    record = {'session_id': session_id}

    record.update(
        create_recency_features(types, days)
    )

    recency_records.append(record)


# Convert to DataFrame
recency_df = pd.DataFrame(recency_records)

print("Recency feature table created!")
print("Shape:", recency_df.shape)

display(recency_df.head())

In [ ]:
# Save session-level recency features

recency_path = OUTPUT_DIR / "session_recency_features.parquet"

recency_df.to_parquet(
    recency_path,
    index=False
)

print("Saved:", recency_path)
print("File exists:", recency_path.exists())

## Create candidate-specific recency features
These will help the model determine whether a user has previously interacted with a particular candidate product

In [ ]:
# STEP 8A — Create exact-product history lookup
def build_product_history_lookup(wids, types, days):

    lookup = defaultdict(lambda: {
        'exact_wid_click_0_7d': 0,
        'exact_wid_cart_0_7d': 0,
        'exact_wid_ord_0_7d': 0,
        'exact_wid_cart_7_90d': 0,
        'exact_wid_ord_7_90d': 0,
        'exact_wid_ord_90_365d': 0,
        'exact_wid_last_seen_days': np.nan
    })

    for wid, action, day in zip(wids, types, days):

        wid = str(wid)
        record = lookup[wid]

        # Most recent interaction with this product
        if np.isnan(record['exact_wid_last_seen_days']):
            record['exact_wid_last_seen_days'] = day
        else:
            record['exact_wid_last_seen_days'] = min(
                record['exact_wid_last_seen_days'], day
            )

        if 0 <= day < 7:
            if action == 'CLICK':
                record['exact_wid_click_0_7d'] += 1
            elif action == 'CART':
                record['exact_wid_cart_0_7d'] += 1
            elif action == 'ORD':
                record['exact_wid_ord_0_7d'] += 1

        elif 7 <= day < 90:
            if action == 'CART':
                record['exact_wid_cart_7_90d'] += 1
            elif action == 'ORD':
                record['exact_wid_ord_7_90d'] += 1

        elif 90 <= day < 365:
            if action == 'ORD':
                record['exact_wid_ord_90_365d'] += 1

    return lookup

In [ ]:
# STEP 8B — Test exact-product history lookup

sample = user_clean.iloc[0]

lookup = build_product_history_lookup(
    sample['history_wid_clean'],
    sample['history_type_clean'],
    sample['history_days_clean']
)

print("Unique historically interacted products:", len(lookup))

# Show first five products
for wid, features in list(lookup.items())[:5]:
    print("\nProduct:", wid)
    print(features)

In [ ]:
# STEP 8C — Test candidate-history matching

candidate_ids = [
    str(wid)
    for wid in sample['candidate_wid_list_parsed']
]

# Avoid creating new entries for unmatched products
matched = [wid for wid in candidate_ids if wid in lookup]

print("Candidates:", len(candidate_ids))
print("Candidates found in history:", len(matched))

if matched:
    print("\nExample matched candidate:", matched[0])
    print("Features:", lookup[matched[0]])
else:
    print("\nNo exact-product matches in this session.")

##Load product_df

In [ ]:
PROJECT_DIR = Path(
    "/content/drive/MyDrive/BT4222 Final/JDSearch"
)

PRODUCT_CHUNKS_DIR = PROJECT_DIR / "Raw Data" / "product_chunks"

product_files = sorted(PRODUCT_CHUNKS_DIR.glob("*.parquet"))

print("Product chunks directory:", PRODUCT_CHUNKS_DIR)
print("Number of product chunks:", len(product_files))

if not product_files:
    print("No Parquet chunks found. Check the directory.")
else:
    print("First chunk:", product_files[0].name)

In [ ]:
#load only wid first to use less ram
assert product_files, "No product chunks found!"

product_parts = []

for file in product_files:
    part = pd.read_parquet(
        file,
        columns=['wid']
    )

    # Standardize product IDs as strings
    part['wid'] = part['wid'].astype('string')

    product_parts.append(part)

product_df = pd.concat(
    product_parts,
    ignore_index=True
)

del product_parts, part
gc.collect()

print("Product metadata loaded successfully!")
print("Shape:", product_df.shape)
print("ID data type:", product_df['wid'].dtype)

display(product_df.head())

## Verify product ID consistency

In [ ]:
# STEP 9A — Check product ID formats

print("CANDIDATE PRODUCT ID")
candidate_id = user_clean['candidate_wid_list_parsed'].iloc[0][0]
print("Value:", repr(candidate_id))
print("Type:", type(candidate_id))

print("\nHISTORICAL PRODUCT ID")
history_id = user_clean['history_wid_clean'].iloc[0][0]
print("Value:", repr(history_id))
print("Type:", type(history_id))

print("\nPRODUCT METADATA")
print("product_df loaded:", 'product_df' in globals())

if 'product_df' in globals():
    product_id = product_df['wid'].iloc[0]
    print("Value:", repr(product_id))
    print("Type:", type(product_id))

## Data Cleaning Summary

In [ ]:
# Save data-cleaning summary
cleaning_summary = pd.DataFrame([
    ["Original sessions", len(user_df)],
    ["Eligible sessions", len(user_clean)],
    ["Excluded all-negative sessions", 2103],
    ["Excluded all-positive sessions", 553],
    ["Total excluded sessions", len(user_df) - len(user_clean)],
    ["Cart actions excluded (>90 days)", 1177],
    ["Candidate-level rows", 15348049],
    ["Candidate chunks validated", len(shuffled_files)],
    ["Training sessions", (user_clean["split"] == "train").sum()],
    ["Validation sessions", (user_clean["split"] == "val").sum()],
    ["Test sessions", (user_clean["split"] == "test").sum()]
], columns=["Metric", "Value"])

display(cleaning_summary)

summary_path = OUTPUT_DIR / "data_cleaning_summary.csv"
cleaning_summary.to_csv(summary_path, index=False)

print("Saved:", summary_path)

##Verify product id Matching

In [ ]:

# Sample 1,000 sessions
sample = user_clean.head(1000)

candidate_ids = set(
    wid
    for ids in sample['candidate_wid_list_parsed']
    for wid in ids
)

history_ids = set(
    wid
    for ids in sample['history_wid_clean']
    for wid in ids
)

# Avoid constructing a huge Python set from all 12M product IDs.
# Check only the sampled IDs using vectorized membership.
sample_ids = candidate_ids | history_ids

matched_ids = set(
    product_df.loc[
        product_df['wid'].isin(sample_ids),
        'wid'
    ]
)

candidate_matches = candidate_ids & matched_ids
history_matches = history_ids & matched_ids

print("CANDIDATE IDS")
print("Unique IDs:", len(candidate_ids))
print("Matched:", len(candidate_matches))
print("Match rate:", f"{len(candidate_matches)/len(candidate_ids):.2%}")

print("\nHISTORICAL IDS")
print("Unique IDs:", len(history_ids))
print("Matched:", len(history_matches))
print("Match rate:", f"{len(history_matches)/len(history_ids):.2%}")

## Save candidate-specific recency features

In [ ]:
# Generate candidate-specific recency features
# Memory-efficient: one chunk at a time

RECENCY_CHUNKS_DIR = OUTPUT_DIR / "candidate_recency_chunks"
RECENCY_CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

shuffled_files = sorted(SHUFFLED_DIR.glob("*.parquet"))

# Use session_id as index for efficient history lookup
history_source = user_clean.set_index('session_id')[
    ['history_wid_clean', 'history_type_clean', 'history_days_clean']
]

for file in tqdm(shuffled_files, desc="Creating candidate recency features"):

    # Skip completed chunks if restarting after a crash
    output_file = RECENCY_CHUNKS_DIR / file.name

    if output_file.exists():
        continue

    # Read one candidate chunk
    df = pd.read_parquet(
        file,
        columns=['session_id', 'wid']
    )

    # Build lookups only for sessions in this chunk
    session_ids = df['session_id'].unique()

    session_lookups = {}

    for sid in session_ids:

        row = history_source.loc[sid]

        session_lookups[sid] = build_product_history_lookup(
            row['history_wid_clean'],
            row['history_type_clean'],
            row['history_days_clean']
        )

    # Create feature records
    records = []

    for sid, wid in zip(df['session_id'], df['wid']):

        lookup = session_lookups[sid]

        # Use get() to avoid creating entries for unseen products
        features = lookup.get(str(wid))

        if features is None:
            records.append(
                [0, 0, 0, 0, 0, 0, np.nan]
            )
        else:
            records.append([
                features[col] for col in FEATURE_COLS
            ])

    # Convert to DataFrame
    feature_df = pd.DataFrame(
        records,
        columns=FEATURE_COLS
    )

    # Preserve identifiers for joining
    feature_df.insert(0, 'wid', df['wid'].values)
    feature_df.insert(0, 'session_id', df['session_id'].values)

    # Reduce memory usage
    for col in FEATURE_COLS[:-1]:
        feature_df[col] = feature_df[col].astype('int32')

    feature_df['exact_wid_last_seen_days'] = (
        feature_df['exact_wid_last_seen_days'].astype('float32')
    )

    # Save
    feature_df.to_parquet(
        output_file,
        index=False
    )

    del df, feature_df, records, session_lookups
    gc.collect()

print("All candidate recency chunks created!")

In [ ]:
# ============================================================
# STEP 13C — Validate candidate recency chunks
# ============================================================

feature_files = sorted(
    RECENCY_CHUNKS_DIR.glob("*.parquet")
)

assert len(feature_files) == len(shuffled_files), (
    "Some feature chunks are missing!"
)

total_rows = 0
matched_rows = 0

for base_file, feature_file in tqdm(
    zip(shuffled_files, feature_files),
    total=len(shuffled_files),
    desc="Validating candidate recency"
):

    base = pd.read_parquet(
        base_file,
        columns=['session_id', 'wid']
    )

    features = pd.read_parquet(feature_file)

    # Check matching row counts
    assert len(base) == len(features)

    # Check identifiers align row by row
    assert base['session_id'].equals(features['session_id'])
    assert base['wid'].equals(features['wid'])

    # Check counts are non-negative
    assert (
        features[FEATURE_COLS[:-1]] >= 0
    ).all().all()

    total_rows += len(features)

    matched_rows += (
        features['exact_wid_last_seen_days'].notna().sum()
    )

    del base, features

assert total_rows == 15_348_049

print("\nValidation passed!")
print("Total candidate rows:", f"{total_rows:,}")
print("Candidates previously interacted with:", f"{matched_rows:,}")
print("Exact-product history match rate:", f"{matched_rows / total_rows:.2%}")

## Save feature dictionary

In [ ]:
feature_dictionary = pd.DataFrame([
    ["session_id", "Unique search-session identifier; join key, not model feature", "ID"],
    ["click_count_0_7d", "Clicks recorded 0–7 days before query", "Count"],
    ["cart_count_0_7d", "Cart actions recorded 0–7 days before query", "Count"],
    ["ord_count_0_7d", "Orders recorded 0–7 days before query", "Count"],
    ["flw_count_0_7d", "Follow actions recorded 0–7 days before query", "Count"],
    ["cart_count_7_90d", "Cart actions recorded 7–90 days before query", "Count"],
    ["ord_count_7_90d", "Orders recorded 7–90 days before query", "Count"],
    ["flw_count_7_90d", "Follow actions recorded 7–90 days before query", "Count"],
    ["ord_count_90_365d", "Orders recorded 90–365 days before query", "Count"],
    ["history_len_clean", "Historical interactions retained after cart-outlier exclusion", "Count"],
    ["history_span_days", "Days since oldest retained historical interaction", "Days"],
    ["days_since_last_interaction", "Days since most recent retained historical interaction", "Days"],
    ["exact_wid_click_0_7d", "Clicks on this candidate within 7 days", "Count"],
    ["exact_wid_cart_0_7d", "Cart actions on this candidate within 7 days", "Count"],
    ["exact_wid_ord_0_7d", "Orders of this candidate within 7 days", "Count"],
    ["exact_wid_cart_7_90d", "Cart actions on this candidate 7–90 days ago", "Count"],
    ["exact_wid_ord_7_90d", "Orders of this candidate 7–90 days ago", "Count"],
    ["exact_wid_ord_90_365d", "Orders of this candidate 90–365 days ago", "Count"],
    ["exact_wid_last_seen_days", "Days since most recent interaction with this candidate; NaN if unseen", "Days"]
], columns=["Feature", "Description", "Type"])

feature_dictionary.to_csv(
    OUTPUT_DIR / "feature_dictionary.csv",
    index=False
)

print("Feature dictionary saved!")

## Data preprocessing notes
(siqian)
I've completed and validated Person 1's main preprocessing pipeline.

We now have 171,175 eligible sessions, split 80/10/10, and 15,348,049 candidate-level rows saved across 86 shuffled Parquet chunks. All candidate-label pairs were preserved.

I also reconstructed the historical timelines, excluded 1,177 anomalous cart actions from engineered history features, and created both session-level and exact-product recency features using the 0–7, 7–90 and 90–365 day windows.

One interesting finding is that only 0.78% of candidate rows match a product in the user's recorded history, so exact-product features are very sparse. This supports exploring broader brand/category preferences and query-match features.

The files are saved under `BT4222 Final/JDSearch/processed/`.
Please use the saved split and shuffled candidate chunks so we work with consistent datasets.